# BÔNUS LLM — Parte 1: LLM integrado ao pipeline de ML

**Cenário:** o modelo treinado nos notebooks 1 e 2 já prevê o **perfil** do aluno (DIFICULDADE, REGULAR, BOM, MUITO_BOM, EXCELENTE).
Agora vamos usar um **LLM** para transformar essa previsão numa **recomendação de mentoria** em linguagem natural.

```
aluno ──► classificador (Etapa 2, porta 32000) ──► perfil ──► LLM (Ollama) ──► recomendação
```

- O **modelo clássico decide** (é rápido, barato e foi medido com acurácia no MLflow).
- O **LLM explica** (gera o texto para o coordenador).

Ferramentas usadas: Ollama (LLM local em CPU), cliente `openai`, **MLflow Prompt Registry** e o microsserviço `mentoria` (porta 32001).

In [ ]:
# Configuração: as URLs vêm do docker-compose (variáveis de ambiente do container)
import os, time, json, requests
import pandas as pd
import mlflow
from openai import OpenAI

MLFLOW_URI = os.environ["MLFLOW_TRACKING_URI"]        # http://mlflow:5000
CLASSIFICADOR_URL = os.environ["CLASSIFICADOR_URL"]   # http://classificador:5000
MENTORIA_URL = os.environ["MENTORIA_URL"]             # http://mentoria:5000
LLM_MODELO = os.environ["LLM_MODELO"]                 # qwen3:1.7b

mlflow.set_tracking_uri(MLFLOW_URI)
llm = OpenAI()   # usa OPENAI_BASE_URL=http://ollama:11434/v1 (API compatível com a da OpenAI)

IP = requests.get("http://checkip.amazonaws.com").text.strip()
print("MLflow UI     : http://%s:8089" % IP)
print("Mentoria (API): http://%s:32001" % IP)
print("Modelo LLM    :", LLM_MODELO)

## 1. Conhecendo o LLM local

O Ollama roda o modelo **na CPU da própria VM** (sem GPU e sem custo de API).
O código abaixo é o mesmo que seria usado com a OpenAI, Bedrock (via gateway) ou outro provedor: só muda a URL.

In [ ]:
# Quais modelos o Ollama baixou?
[m.id for m in llm.models.list().data]

In [ ]:
inicio = time.time()
resp = llm.chat.completions.create(
    model=LLM_MODELO,
    messages=[{"role": "user", "content": "Em uma frase, o que é MLOps?"}],
    max_tokens=100,
    extra_body={"reasoning_effort": "none"},   # desliga o modo "pensar" do qwen3 (veremos o custo disso na Parte 2)
)
print(resp.choices[0].message.content)
print(f"\nTempo: {time.time() - inicio:.1f}s | tokens de entrada: {resp.usage.prompt_tokens} | tokens de saída: {resp.usage.completion_tokens}")

> **Responda:**
> - Quanto tempo levou? Quantos tokens foram gerados por segundo?
> - Por que o tempo da primeira chamada costuma ser maior que o das seguintes?
> - Em um provedor pago, o que define o custo dessa chamada?

## 2. Os dados do aluno (e os dados pessoais)

In [ ]:
alunos = pd.read_csv("avaliacao_mentoria.csv")
alunos.head(3)

> **Responda:** quais colunas são **dados pessoais** (PII) e não deveriam ser enviadas a um LLM, ainda mais a um LLM de terceiros?

## 3. ML clássico: perguntando o perfil ao classificador da Etapa 2

É o **mesmo microsserviço** (`predict.py` + `model.joblib`) que você publicou na porta 32000.
Aqui ele é chamado pelo nome do serviço na rede do docker-compose: `http://classificador:5000`.

In [ ]:
FEATURES = [
    "REPROVACOES_MAT_1", "REPROVACOES_MAT_2", "REPROVACOES_MAT_3", "REPROVACOES_MAT_4",
    "NOTA_MAT_1", "NOTA_MAT_2", "NOTA_MAT_3", "NOTA_MAT_4",
    "INGLES", "H_AULA_PRES", "TAREFAS_ONLINE", "FALTAS",
]

def classificar(aluno):
    corpo = {"fields": FEATURES, "values": [float(aluno[f]) for f in FEATURES]}
    return requests.post(f"{CLASSIFICADOR_URL}/predict", json=corpo, timeout=10).json()["_PERFIL_CALCULADO"]

aluno = alunos.iloc[0].to_dict()
perfil = classificar(aluno)
print("Perfil previsto:", perfil, "| perfil real:", aluno["PERFIL_REAL"])

## 4. Mascarando dados pessoais antes do LLM

Regra de ouro: **o LLM só recebe o que precisa** para a tarefa. Nome e matrícula não ajudam a recomendar mentoria.

In [ ]:
CAMPOS_PII = ["NOME", "MATRICULA"]

def mascarar_pii(aluno):
    return {k: v for k, v in aluno.items() if k not in CAMPOS_PII + ["PERFIL_REAL"]}

aluno_limpo = mascarar_pii(aluno)
print("Antes :", {k: aluno[k] for k in CAMPOS_PII})
print("Depois:", [k for k in CAMPOS_PII if k in aluno_limpo] or "nenhum dado pessoal")

> **Exercício:** imagine que enviássemos a linha inteira do CSV ao LLM de um provedor externo.
> O que vazaria? Que lei brasileira trata disso? Onde mais esses dados poderiam ficar gravados (logs, traces, cache)?

## 5. Primeiro prompt (escrito direto no código)

In [ ]:
dados = ", ".join(f"{k}={aluno_limpo[k]}" for k in FEATURES)
mensagens = [
    {"role": "system", "content": "Você é um coordenador pedagógico de uma faculdade. Com base no perfil previsto por um "
                                  "modelo de Machine Learning e nos dados do aluno, escreva em português do Brasil uma "
                                  "recomendação de mentoria curta, citando o perfil e sugerindo 2 ações concretas."},
    {"role": "user", "content": f"Perfil previsto: {perfil}\nDados do aluno: {dados}"},
]
inicio = time.time()
resp = llm.chat.completions.create(model=LLM_MODELO, messages=mensagens, max_tokens=250, temperature=0.3,
                                   extra_body={"reasoning_effort": "none"})
print(resp.choices[0].message.content)
print(f"\n({time.time() - inicio:.1f}s, {resp.usage.completion_tokens} tokens)")

> **Responda:** o texto respeitou o pedido (perfil citado, 2 ações)? Ele **inventou** algo que não está nos dados?
> Se você mudar o prompt no código, como saberia depois qual versão gerou cada resposta?

## 6. Prompt como artefato versionado: MLflow Prompt Registry

Assim como os **modelos** foram registrados no MLflow (bônus anterior), o **prompt** também vira um artefato com versão, autor e histórico.
O microsserviço `mentoria` não tem o prompt no código: ele carrega `prompts:/mentoria@producao` do MLflow.

In [ ]:
PROMPT_V1 = [
    {"role": "system", "content": "Você é um coordenador pedagógico de uma faculdade. Com base no perfil previsto por um "
                                  "modelo de Machine Learning e nos dados do aluno, escreva em português do Brasil uma "
                                  "recomendação de mentoria curta, citando o perfil e sugerindo 2 ações concretas."},
    {"role": "user", "content": "Perfil previsto: {{perfil}}\nDados do aluno: {{dados}}"},
]

prompt = mlflow.genai.register_prompt(
    name="mentoria",
    template=PROMPT_V1,
    commit_message="v1: prompt inicial do notebook 3",
    tags={"etapa": "bonus-llm", "autor": "aluno"},
)
mlflow.genai.set_prompt_alias("mentoria", alias="producao", version=prompt.version)
print(f"Registrado: {prompt.name} versão {prompt.version} (alias 'producao')")

**No MLflow** (`http://IP:8089`), abra a aba **Prompts** → `mentoria` e veja a versão, o template e o alias `producao`.

In [ ]:
# Carregando o prompt pelo alias, como o microsserviço faz
p = mlflow.genai.load_prompt("prompts:/mentoria@producao")
mensagens = p.format(perfil=perfil, dados=dados)
resp = llm.chat.completions.create(model=LLM_MODELO, messages=mensagens, max_tokens=250, temperature=0.3,
                                   extra_body={"reasoning_effort": "none"})
print(f"[{p.name} v{p.version}]\n")
print(resp.choices[0].message.content)

## 7. O pipeline completo como microsserviço

O serviço `mentoria` junta tudo: **mascarar PII → classificar → carregar o prompt → chamar o LLM**.
Repare que enviamos o aluno **com** nome e matrícula: o serviço remove esses campos antes de qualquer outra etapa.

In [ ]:
aluno_com_pii = {k: (v.item() if hasattr(v, "item") else v) for k, v in alunos.iloc[1].to_dict().items() if k != "PERFIL_REAL"}
r = requests.post(f"{MENTORIA_URL}/mentoria", json={"aluno": aluno_com_pii, "janela": "teste_b1"}, timeout=300).json()
print(json.dumps(r, indent=2, ensure_ascii=False))

Teste também **pelo terminal da VM** (fora do Jupyter), usando a porta publicada 32001:

```bash
curl -s http://localhost:32001/ | python3 -m json.tool
curl -s -X POST http://localhost:32001/mentoria -H "Content-Type: application/json" \
  -d '{"aluno": {"NOME": "Fulano", "MATRICULA": 1, "REPROVACOES_MAT_1": 1, "REPROVACOES_MAT_2": 1,
       "REPROVACOES_MAT_3": 1, "REPROVACOES_MAT_4": 1, "NOTA_MAT_1": 4.3, "NOTA_MAT_2": 4.0, "NOTA_MAT_3": 3.1,
       "NOTA_MAT_4": 4.9, "INGLES": 0, "H_AULA_PRES": 3, "TAREFAS_ONLINE": 4, "FALTAS": 3}}' | python3 -m json.tool
```

> **Perguntas finais da Parte 1:**
> 1. Por que não deixar o LLM classificar o aluno sozinho? (pense em custo, tempo, previsibilidade e em como medir a acurácia)
> 2. Que dados nunca devem chegar a um LLM? Em que ponto do pipeline o mascaramento deve acontecer?
> 3. Qual a vantagem de o serviço carregar o prompt do Registry (`@producao`) em vez de tê-lo fixo no código?
> 4. O campo `prompt_versao` da resposta mostra qual versão gerou o texto. Por que isso é importante numa auditoria?